In [0]:
from pyspark.sql.types import *
from pyspark.sql import functions as f,Window
from pyspark.sql.functions import regexp_replace


customer_data = [
    (101,' Rahul ','RAHUL@GMAIL.COM ',' Bangalore ','Gold','98765-43210','2025-01-02'),
    (101,' Rahul ','rahul@gmail.com','Bangalore','Gold','98765-43210','2025-01-01'),
    (102,'Amit','amit@gmail.com','Pune','Silver','98765-55210','2025-01-05'),
    (103,'John','johngmail.com','Chennai','Gold','98765-00210','2025-01-04'),
    (104,None,'david@gmail.com','Mumbai','Bronze','98765-00910','2025-01-03'),
    (104,' DAVID ','DAVID@GMAIL.COM ',' Mumbai ','Platinum','98765-00910','2025-01-10'),
    (105,'Sara','sara@gmail.com',None,'Silver','99865-88810','2025-01-14'),
    (106,'Kiran','kiran@gmail.com','Hyderabad','Gold','9876543210','2025-01-08'),
    (107,'Anu','anu@gmail.com','Kolkata','Gold','9876555510','2025-01-06'),
    (108,'Rohit','rohit@gmail.com','Delhi','Bronze','9876541111','2025-01-05'),
    (109,'Priya','PRIYA@GMAIL.COM ',' Chennai ','Silver','9876542222','2025-01-06'),
    (110,'Vijay','vijay@gmail.com','Pune','Gold','9876543333','2025-01-08'),
    (111,'Sneha','sneha@gmail.com','Mumbai','Platinum','9876544444','2025-01-10'),
    (112,'Arjun','arjun@gmail.com','Hyderabad','Silver','9876545555','2025-01-09'),
    (113,'Deepa','deepa@gmail.com','Bangalore','Gold','9876546666','2025-01-12'),
    (114,'Manoj','manoj@gmail.com','Delhi','Bronze','9876547777','2025-01-11'),
    (115,'Neha','neha@gmail.com','Pune','Silver','9876548888','2025-01-07'),
    (116,'Varun','varun@gmail.com','Mumbai','Gold','9876549999','2025-01-06'),
    (117,'Pooja','pooja@gmail.com','Chennai','Gold','9876500000','2025-01-05'),
    (118,'Ramesh',None,'Mysore','Gold','9876511111','2025-01-05'),
    (119,'Ajay','ajay@gmail.com','Jaipur','Silver','9876522222','2025-01-09'),
    (120,'Meena','meena@gmail.com','Lucknow','Bronze','9876533333','2025-01-10'),
    (121,'Akash','akash@gmail.com','Nagpur','Gold','9876544445','2025-01-08'),
    (122,'Divya','divya@gmail.com','Indore','Silver','9876545556','2025-01-07'),
    (123,'Kavya','kavya@gmail.com','Surat','Gold','9876546667','2025-01-06'),
    (124,'Harish','harish@gmail.com','Patna','Bronze','9876547778','2025-01-05'),
    (125,'Nisha','nisha@gmail.com','Bhopal','Gold','9876548889','2025-01-04'),
    (126,'Suresh','suresh@gmail.com','Kochi','Silver','9876549990','2025-01-03'),
    (127,'Lakshmi','lakshmi@gmail.com','Vizag','Gold','9876551111','2025-01-02'),
    (128,'Aravind','aravind@gmail.com','Coimbatore','Gold','9876552222','2025-01-01')
]

orders_data = [
    (1001,101,201,301,50000,'2025-01-03'),
    (1002,101,202,301,1200,'2025-01-04'),
    (1003,101,203,301,2500,'2025-01-05'),
    (1004,101,206,301,9000,'2025-01-06'),
    (1005,102,204,302,18000,'2025-01-05'),
    (1006,102,202,302,1200,'2025-01-06'),
    (1007,103,205,304,15000,'2025-01-06'),
    (1008,104,201,303,52000,'2025-01-11'),
    (1009,104,207,303,35000,'2025-01-12'),
    (1010,105,208,305,25000,'2025-01-15'),
    (1011,106,209,305,75000,'2025-01-08'),
    (1012,106,202,305,1200,'2025-01-09'),
    (1013,107,201,307,48000,'2025-01-07'),
    (1014,108,204,306,18000,'2025-01-06'),
    (1015,109,206,304,9000,'2025-01-07'),
    (1016,110,201,302,51000,'2025-01-08'),
    (1017,110,202,302,1200,'2025-01-09'),
    (1018,110,203,302,2500,'2025-01-10'),
    (1019,111,209,303,82000,'2025-01-10'),
    (1020,112,203,305,2500,'2025-01-09'),
    (1021,113,202,301,1200,'2025-01-12'),
    (1022,114,205,306,15000,'2025-01-11'),
    (1023,115,201,302,49000,'2025-01-08'),
    (1024,115,202,302,1200,'2025-01-09'),
    (1025,115,206,302,9000,'2025-01-10'),
    (1026,116,210,303,65000,'2025-01-06'),
    (1027,117,206,304,9500,'2025-01-05'),
    (1028,118,209,310,76000,'2025-01-06'),
    (1029,119,207,308,34000,'2025-01-09'),
    (1030,120,204,309,17000,'2025-01-10'),
    (1031,121,201,317,53000,'2025-01-08'),
    (1032,121,202,317,1200,'2025-01-09'),
    (1033,122,202,314,1200,'2025-01-07'),
    (1034,123,203,315,2600,'2025-01-06'),
    (1035,124,205,313,16000,'2025-01-05'),
    (1036,125,201,318,54000,'2025-01-04'),
    (1037,126,206,319,9200,'2025-01-03'),
    (1038,127,209,311,78000,'2025-01-02'),
    (1039,128,201,312,50000,'2025-01-01'),
    # Null customer_id
    (1040,None,202,301,1200,'2025-01-05'),
    # Customer doesn't exist
    (1041,999,201,301,55000,'2025-01-05'),
    # Product doesn't exist
    (1042,110,999,302,25000,'2025-01-05'),
    # Store doesn't exist
    (1043,111,201,999,50000,'2025-01-05'),
    # Negative amount
    (1044,112,201,305,-50000,'2025-01-05'),
    # Future order
    (1045,113,209,301,82000,'2030-01-01'),
    # Exact duplicate
    (1046,114,205,306,15000,'2025-01-11'),
    (1046,114,205,306,15000,'2025-01-11')
]

customers_schema = StructType(
    [
        StructField('customer_id',IntegerType()),
        StructField('customer_name',StringType()),
        StructField('customer_email',StringType()),
        StructField('customer_city',StringType()),
        StructField('segment',StringType()),
        StructField('customer_phone',StringType()),
        StructField('load_dt',StringType())
    ]
)

orders_schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("customer_id", IntegerType()),
    StructField("product_id", IntegerType()),
    StructField("store_id", IntegerType()),
    StructField("amount", IntegerType()),
    StructField("order_dt", StringType())
])

product_data = [
    (201,'Laptop','Electronics','Dell',50000,'Y'),
    (202,'Mouse','Accessories','Logitech',1200,'Y'),
    (203,'Keyboard','Accessories','Logitech',2500,'Y'),
    (204,'Monitor','Electronics','Samsung',18000,'Y'),
    (205,'Printer','Electronics','HP',15000,'Y'),
    (206,'SSD','Storage','Samsung',9000,'Y'),
    (207,'Camera','Electronics','Canon',35000,'Y'),
    (208,'Tablet','Electronics','Apple',25000,'Y'),
    (209,'TV','Electronics','Sony',75000,'Y'),
    (210,'Mobile','Electronics','Apple',65000,'Y'),
    # Duplicate Product
    (210,'Mobile','Electronics','Apple',65000,'Y'),
    # Null Category
    (211,'Router',None,'TP-Link',3500,'Y'),
    # Dirty Product Name
    (212,' laptop ','Electronics','Lenovo',48000,'Y'),
    # Lower Case
    (213,'mouse','Accessories','HP',900,'Y'),
    # Null Brand
    (214,'Speaker','Electronics',None,4500,'Y'),
    # Inactive Products
    (215,'Scanner','Electronics','Canon',8000,'N'),
    (216,'Projector','Electronics','Epson',45000,'N'),
    # Null Price
    (217,'Pendrive','Storage','SanDisk',None,'Y'),
    # Invalid Price
    (218,'Hard Disk','Storage','WD',-5000,'Y'),
    # Duplicate
    (219,'Webcam','Accessories','Logitech',3200,'Y'),
    (219,'Webcam','Accessories','Logitech',3200,'Y')
]

product_schema = StructType([
    StructField("product_id", IntegerType()),
    StructField("product_name", StringType()),
    StructField("category", StringType()),
    StructField("brand", StringType()),
    StructField("price", IntegerType()),
    StructField("active", StringType())
])

store_data = [
    (301,'Bangalore Central','Bangalore','South','Rajesh'),
    (302,'Pune West','Pune','West','Amit'),
    (303,'Mumbai Mall','Mumbai','West','Kiran'),
    (304,'Chennai Plaza','Chennai','South','Ravi'),
    (305,'Hyderabad Hub','Hyderabad','South','Prakash'),
    (306,'Delhi Center','Delhi','North','Arun'),
    (307,'Kolkata Point','Kolkata','East','Sanjay'),
    (308,'Jaipur Store','Jaipur','North','Nitin'),
    (309,'Lucknow Store','Lucknow','North','Rahul'),
    (310,'Mysore Outlet','Mysore','South','Suresh'),
    # Duplicate
    (310,'Mysore Outlet','Mysore','South','Suresh'),
    # Null Region
    (311,'Vizag Store','Vizag',None,'Mahesh'),
    # Dirty City
    (312,'Coimbatore Mall',' coimbatore ','South','Lokesh'),
    # Null Manager
    (313,'Patna Center','Patna','East',None),
    # Lower Case
    (314,'Indore Hub','indore','Central','Ramesh'),
    # Mixed Case
    (315,'Surat Plaza',' SURAT ','West','Kishore'),
    # Duplicate Store Name Different Case
    (316,'bangalore central',' Bangalore ','South','Rajesh'),
    # Additional Stores
    (317,'Nagpur Center','Nagpur','Central','Deepak'),
    (318,'Bhopal Plaza','Bhopal','Central','Harish'),
    (319,'Kochi Store','Kochi','South','Joseph'),
    (320,'Goa Outlet','Goa','West','Vinay')
]

store_schema = StructType([
    StructField("store_id", IntegerType()),
    StructField("store_name", StringType()),
    StructField("city", StringType()),
    StructField("region", StringType()),
    StructField("manager", StringType())
])

bronze_customers = spark.createDataFrame(customer_data,customers_schema)
bronze_orders = spark.createDataFrame(orders_data,orders_schema)
bronze_products = spark.createDataFrame(product_data,product_schema)
bronze_stores = spark.createDataFrame(store_data,store_schema)

# Drop Null records
bronze_customers = bronze_customers.dropna(subset=('customer_id'))
bronze_orders = bronze_orders.dropna(subset=('customer_id'))
bronze_products = bronze_products.filter(f.col('product_id').isNotNull())
bronze_stores = bronze_stores.filter(f.col('store_id').isNotNull())

# Drop duplicates
bronze_customers = bronze_customers.dropDuplicates()
bronze_orders = bronze_orders.dropDuplicates()
bronze_products = bronze_products.dropDuplicates()
bronze_stores = bronze_stores.dropDuplicates()

# Time Stamp Change
bronze_customers = bronze_customers.withColumn('load_dt',f.col('load_dt').cast(DateType()))
bronze_orders = bronze_orders.withColumn('order_dt',f.col('order_dt').cast(DateType()))

# Customer latest
w = Window.partitionBy('customer_id').orderBy(f.col('load_dt').desc())
bronze_customers = bronze_customers.withColumn('rn',f.row_number().over(w)).filter(f.col('rn')==1).drop('rn')

# Customer null handling
bronze_customers = bronze_customers.fillna({
    'customer_name':'unknown',
    'customer_email':'unknown',
    'customer_city':'unknown',
    'segment':'unknown',
    'customer_phone':'unknown'
})

phone_regex = r'[^0-9]'

# Standardize
bronze_customers = bronze_customers.withColumn('customer_name',f.initcap(f.trim('customer_name'))).withColumn('customer_email',f.initcap(f.trim(f.col('customer_email')))).withColumn('customer_city',f.initcap(f.trim(f.col('customer_city')))).withColumn('segment',f.initcap(f.trim(f.col('segment')))).withColumn('customer_phone',f.regexp_replace(f.col('customer_phone'),phone_regex,''))


# Data Validation
email_regex = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'


valid_email = f.col('customer_email').rlike(email_regex)
vaild_phone = f.col('customer_phone').rlike(phone_regex)

bronze_customers = bronze_customers.withColumn('data_quality',f.when(
   (valid_email) & (f.length(f.col('customer_phone')) == 10) &
    (f.col('load_dt') <= f.current_date()),'Pass'
).otherwise('Fail'))

silver_customers = bronze_customers.filter(f.col('data_quality')=='Pass')
error_customers = bronze_customers.filter(f.col('data_quality')=='Fail')

# Orders table
bronze_orders = bronze_orders.withColumn('data_quality',f.when(
    (f.col('order_dt') <= f.current_date()) &
    (f.col('amount') >= 0)
    ,'Pass').otherwise('Fail'))
silver_orders = bronze_orders.filter(f.col('data_quality')=='Pass')

bronze_products = bronze_products.fillna({
    'category':'unknown',
    'brand':'unknown',
    'price':0,
    'active':'N'
})

# Products table
bronze_products = bronze_products.withColumn('data_quality',f.when(
    (f.col('price') >= 0) & (f.col('active') == 'Y')
    ,'Pass').otherwise('Fail'))

silver_products = bronze_products.filter(f.col('data_quality')=='Pass')

# Stores table
bronze_stores = bronze_stores.fillna({
    'store_name':'unknown',
    'city':'unknown',
    'region':'unknown',
    'manager':'unknown'
})

silver_stores = bronze_stores.withColumn('store_name',f.initcap(f.trim('store_name'))).withColumn('city',f.initcap(f.trim(f.col('city')))).withColumn('region',f.initcap(f.trim(f.col('region')))).withColumn('manager',f.initcap(f.trim(f.col('manager'))))

# silver_stores.persist()
# silver_products.persist()
# silver_orders.persist()
# silver_customers.persist()

# Look Up Tables

gold_orders = (silver_orders.join(silver_customers,on='customer_id',how='left').join(silver_products,on='product_id',how='left').join(silver_stores,on='store_id',how='left')).select('customer_id','customer_name','customer_email','customer_city','segment','customer_phone','order_id','product_id','store_id','amount','product_name','category','brand','price','active','store_name',f.col('city').alias('store_city'),f.col('region').alias('store_region'),f.col('manager').alias('store_manager'))

gold_orders = gold_orders.dropna()

# gold_orders.show()
gold_orders.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('gold_orders')



In [0]:
%sql


select customer_id,customer_name,sum(amount) as total_sales
from gold_orders group by customer_id,customer_name
having total_sales > 50000;

select customer_id,customer_name,count(order_id) as ordersCount from gold_orders
group by customer_id,customer_name
having ordersCount > 2;

with src as (
    select customer_id,customer_name,amount,dense_rank() over(order by amount desc) as rn
    from gold_orders
)
select customer_id,customer_name,amount from src where rn <=5;

with src1 as (
    select product_name,store_city,count(order_id) as ordersCount
    from gold_orders
    group by product_name,store_city
)
,src2 as (
    select product_name,store_city,ordersCount,row_number() over(partition by store_city order by ordersCount desc) as rn
    from src1
)
select product_name,store_city from src2 where rn == 1;

select customer_id,count(distinct product_id) as distinctProducts from gold_orders
group by customer_id
having distinctProducts > 2;

select store_city,sum(amount) as revenue
from gold_orders
group by store_city order by store_city;

select store_region,sum(amount) as revenue
from gold_orders
group by store_region order by store_region;

select customer_id,avg(amount) as avg from gold_orders
group by customer_id order by avg desc;

select distinct customer_id
from gold_orders
where amount > 20000






In [0]:
%sql

select customer_id,customer_name,segment,customer_city,count(order_id) as total_orders,sum(amount) as total_sales,
avg (order_id) as average_order,max(amount) as highest_order
from gold_orders
group by customer_id,customer_name,segment,customer_city;
